In [1]:
import json
import random
import collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt 
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

### Config

In [2]:
SEED         = 42
DATA_PATH    = "train.json"
TEST_PATH    = "test.json"
MAX_SEQ_LEN  = 64
EMBEDDING_DIM= 128
HIDDEN_DIM   = 128    # mỗi chiều → output BiLSTM = 256
NUM_LAYERS   = 2
DROPOUT      = 0.3
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-3

NER_LABELS = [
    "O",
    "B-PERSON",              "I-PERSON",
    "B-ORGANIZATION",        "I-ORGANIZATION",
    "B-LOCATION",            "I-LOCATION",
]
SENT_LABELS = ["neg", "neu", "pos"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


### Load data

In [3]:
def load_data(path):
    """
    Đọc file JSONL — mỗi dòng có dạng:
      {"tokens": [...], "ner_tags": [...], "sentiment": "pos|neg|neu"}
    """
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            r         = json.loads(line)
            tokens     = [w.lower() for w in r["tokens"]]
            ner_labels = r["ner_tags"]
            sentiment  = r.get("sentiment", "neu")

            if sentiment not in SENT_LABELS:
                print(f"  [skip] sentiment không hợp lệ: {sentiment}")
                continue

            valid_ner  = set(NER_LABELS)
            ner_labels = [l if l in valid_ner else "O" for l in ner_labels]

            data.append({
                "tokens"    : tokens,
                "ner_labels": ner_labels,
                "sentiment" : sentiment,
            })

    from collections import Counter
    dist = Counter(s["sentiment"] for s in data)
    print(f"Loaded {len(data)} samples | Sentiment: {dict(dist)}")
    return data


def tokenize(text):
    """
    Tokenize văn bản đầu vào cho inference.
    Ưu tiên underthesea, fallback về tách khoảng trắng.
    """
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(text)
        return [t.replace(" ", "_").lower() for t in tokens]
    except ImportError:
        return text.lower().split()

### Vocabulary

In [4]:
class Vocabulary:
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self):
        self.w2i = {"<PAD>": 0, "<UNK>": 1}
        self.i2w = {0: "<PAD>", 1: "<UNK>"}

    def build(self, data):
        freq = collections.Counter()
        for s in data:
            freq.update(s["tokens"])
        for w in freq:
            if w not in self.w2i:
                i = len(self.w2i)
                self.w2i[w] = i
                self.i2w[i] = w
        print(f"Vocab size: {len(self.w2i)}")

    def encode(self, tokens):
        return [self.w2i.get(t, self.UNK_IDX) for t in tokens]

    def __len__(self):
        return len(self.w2i)

### Dataset & DataLoader

In [5]:
class NERSentDataset(Dataset):
    def __init__(self, data, vocab):
        self.data  = data
        self.vocab = vocab
        self.ner_l2i  = {l: i for i, l in enumerate(NER_LABELS)}
        self.sent_l2i = {l: i for i, l in enumerate(SENT_LABELS)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        s      = self.data[idx]
        tokens = s["tokens"][:MAX_SEQ_LEN]
        labels = s["ner_labels"][:MAX_SEQ_LEN]
        labels = [l if l in self.ner_l2i else "O" for l in labels]

        tok_ids  = torch.tensor(self.vocab.encode(tokens), dtype=torch.long)
        ner_ids  = torch.tensor([self.ner_l2i[l] for l in labels], dtype=torch.long)
        sent_id  = torch.tensor(self.sent_l2i[s["sentiment"]], dtype=torch.long)
        return tok_ids, ner_ids, sent_id, len(tokens)


def collate_fn(batch):
    tok_list, ner_list, sent_list, lens = zip(*batch)

    # Sắp xếp giảm dần theo length (yêu cầu của pack_padded_sequence)
    order    = sorted(range(len(lens)), key=lambda i: lens[i], reverse=True)
    tok_list = [tok_list[i] for i in order]
    ner_list = [ner_list[i] for i in order]
    sent_ids = torch.stack([sent_list[i] for i in order])
    lens     = torch.tensor([lens[i] for i in order], dtype=torch.long)

    tok_ids = pad_sequence(tok_list, batch_first=True, padding_value=0)
    ner_ids = pad_sequence(ner_list, batch_first=True, padding_value=-100)
    return tok_ids, ner_ids, sent_ids, lens

### CRF

In [6]:
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags) * 0.1)

    def _log_partition(self, emissions, mask):
        """
        Tính log-partition function Z bằng forward algorithm.
        emissions : (B, T, N)
        mask      : (B, T)  — True = token thật, False = PAD
        """
        B, T, N = emissions.shape
        alpha = emissions[:, 0, :]                          # (B, N)

        for t in range(1, T):
            alpha_t = alpha.unsqueeze(2) + self.transitions  # (B, N, N)
            alpha_t = torch.logsumexp(alpha_t, dim=1)        # (B, N)
            alpha_t = alpha_t + emissions[:, t, :]           # (B, N)
            alpha = torch.where(mask[:, t].unsqueeze(1), alpha_t, alpha)

        return torch.logsumexp(alpha, dim=1)                 # (B,)

    def _score_sequence(self, emissions, tags, mask):
        """
        Tính score của chuỗi nhãn đúng (ground truth).
        emissions : (B, T, N)
        tags      : (B, T)
        mask      : (B, T)
        """
        B, T, N = emissions.shape
        score = torch.zeros(B, device=emissions.device)

        score += emissions[:, 0, :].gather(1, tags[:, 0].unsqueeze(1)).squeeze(1)

        for t in range(1, T):
            active = mask[:, t]                              # (B,)
            trans  = self.transitions[tags[:, t-1], tags[:, t]]  # (B,)
            emit   = emissions[:, t, :].gather(1, tags[:, t].unsqueeze(1)).squeeze(1)
            score  += torch.where(active, trans + emit, torch.zeros_like(trans))

        return score                                         # (B,)

    def neg_log_likelihood(self, emissions, tags, lengths):
        """
        Loss = -( score(đúng) - log Z )   → tối thiểu hóa
        emissions : (B, T, N)
        tags      : (B, T)   — padding dùng 0 (sẽ bị mask)
        lengths   : (B,)
        """
        B, T, _ = emissions.shape
        lengths = lengths.to(emissions.device)
        mask = torch.arange(T, device=emissions.device).unsqueeze(0) < lengths.unsqueeze(1)  # (B, T)

        safe_tags = tags.clone()
        safe_tags[safe_tags == -100] = 0

        log_Z    = self._log_partition(emissions, mask)      # (B,)
        gold_score = self._score_sequence(emissions, safe_tags, mask)  # (B,)
        return (log_Z - gold_score).mean()

    def decode(self, emissions, lengths):
        """
        Viterbi decoding — trả về list các chuỗi nhãn tốt nhất.
        emissions : (B, T, N)
        lengths   : (B,)
        """
        B, T, N = emissions.shape
        lengths = lengths.cpu()
        best_paths = []

        for b in range(B):
            L = lengths[b].item()
            emit = emissions[b, :L, :]                      # (L, N)

            viterbi  = emit[0].clone()                       # (N,)
            backptr  = []

            for t in range(1, L):
                v_t    = viterbi.unsqueeze(1) + self.transitions  # (N, N)
                best_s, best_tag = v_t.max(dim=0)            # (N,), (N,)
                backptr.append(best_tag)
                viterbi = best_s + emit[t]

            # Traceback
            best_last = viterbi.argmax().item()
            path = [best_last]
            for bp in reversed(backptr):
                path.append(bp[path[-1]].item())
            path.reverse()
            best_paths.append(path)

        return best_paths


### Model

In [7]:
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, h, pad_mask):
        # h: (B, T, H)  pad_mask: (B, T) True=PAD
        scores = self.attn(h).squeeze(-1)               # (B, T)
        scores = scores.masked_fill(pad_mask, -1e9)
        w      = F.softmax(scores, dim=-1)              # (B, T)
        return torch.bmm(w.unsqueeze(1), h).squeeze(1)  # (B, H)


class BiLSTMCRFNERSent(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        out_dim = HIDDEN_DIM * 2  # bidirectional

        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)
        self.emb_drop  = nn.Dropout(DROPOUT)

        self.bilstm = nn.LSTM(
            EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS,
            batch_first=True, bidirectional=True,
            dropout=DROPOUT if NUM_LAYERS > 1 else 0.0,
        )
        self.lstm_drop = nn.Dropout(DROPOUT)
        self.attention = AttentionPooling(out_dim)

        # NER: linear
        self.ner_linear = nn.Linear(out_dim, len(NER_LABELS))

        # CRF layer
        self.crf = CRF(len(NER_LABELS))

        # Sentiment head
        self.sent_head = nn.Sequential(
            nn.Linear(out_dim, out_dim // 2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(out_dim // 2, len(SENT_LABELS)),
        )

    def _encode(self, tok_ids, lengths):
        emb = self.emb_drop(self.embedding(tok_ids))
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=True)
        packed_out, _ = self.bilstm(packed)
        h, _ = pad_packed_sequence(packed_out, batch_first=True,
                                    total_length=tok_ids.size(1))     # (B, T, 2H)
        return self.lstm_drop(h)

    def forward(self, tok_ids, lengths):
        h = self._encode(tok_ids, lengths)
        emissions = self.ner_linear(h)                               # (B, T, N_ner)
        pad_mask = (tok_ids == 0)
        doc_vec = self.attention(h, pad_mask)
        sent_logits = self.sent_head(doc_vec)                         # (B, N_sent)
        return emissions, sent_logits

    def decode(self, tok_ids, lengths):
        h         = self._encode(tok_ids, lengths)
        emissions = self.ner_linear(h)
        return self.crf.decode(emissions, lengths)   

### Metrics

In [8]:
def spans(bio_seq):
    result, start, typ = set(), None, None
    for i, label in enumerate(bio_seq):
        if label.startswith("B-"):
            if start is not None:
                result.add((start, i - 1, typ))
            start, typ = i, label[2:]
        elif not (label.startswith("I-") and typ == label[2:]):
            if start is not None:
                result.add((start, i - 1, typ))
            start = None
    if start is not None:
        result.add((start, len(bio_seq) - 1, typ))
    return result


def ner_counts(pred_ids, gold_ids):
    tp = fp = fn = 0
    for pred_row, gold_row in zip(pred_ids.tolist(), gold_ids.tolist()):
        g_seq = [NER_LABELS[g] if g != -100 else "O" for g in gold_row]
        p_seq = [NER_LABELS[p] if g != -100 else "O" for p, g in zip(pred_row, gold_row)]
        ps, gs = spans(p_seq), spans(g_seq)
        tp += len(ps & gs)
        fp += len(ps - gs)
        fn += len(gs - ps)
    return tp, fp, fn


### Train

In [9]:
def train(epochs, model, train_loader, val_loader, optimizer, scheduler):
    from collections import Counter
    counts  = Counter(s["sentiment"] for s in train_loader.dataset.data)
    n_class = len(SENT_LABELS)
    weights = torch.tensor(
        [1.0 / (counts.get(l, 1) ** 0.5) for l in SENT_LABELS],
        dtype=torch.float
    )
    weights = (weights / weights.sum() * n_class).to(DEVICE)
    print(f"Sent class weights: { {SENT_LABELS[i]: f'{weights[i].item():.3f}' for i in range(n_class)} }")

    sent_loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

    best_score = -1.0
    best_state = None

    train_losses = []
    val_losses   = []

    for epoch in range(1, epochs + 1):

        model.train()
        train_loss = 0.0
        for tok, ner, sent, lens in train_loader:
            tok, ner, sent = tok.to(DEVICE), ner.to(DEVICE), sent.to(DEVICE)

            emissions, sent_logits = model(tok, lens)

            safe_ner = ner.clone()
            safe_ner[safe_ner == -100] = 0
            ner_loss  = model.crf.neg_log_likelihood(emissions, safe_ner, lens)
            sent_loss = sent_loss_fn(sent_logits, sent)
            loss      = ner_loss + sent_loss

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        all_sp, all_sg = [], []
        ner_tp = ner_fp = ner_fn = 0

        with torch.no_grad():
            for tok, ner, sent, lens in val_loader:
                tok, ner, sent = tok.to(DEVICE), ner.to(DEVICE), sent.to(DEVICE)

                emissions, sent_logits = model(tok, lens)

                safe_ner = ner.clone()
                safe_ner[safe_ner == -100] = 0
                ner_loss  = model.crf.neg_log_likelihood(emissions, safe_ner, lens)
                sent_loss = sent_loss_fn(sent_logits, sent)
                val_loss += (ner_loss + sent_loss).item()

                all_sp.append(sent_logits.argmax(-1).cpu())
                all_sg.append(sent.cpu())

                pred_paths = model.crf.decode(emissions, lens)
                max_len = ner.size(1)
                pred_tensor = torch.zeros_like(ner)
                for b, path in enumerate(pred_paths):
                    pred_tensor[b, :len(path)] = torch.tensor(path)
                tp, fp, fn = ner_counts(pred_tensor.cpu(), ner.cpu())
                ner_tp += tp; ner_fp += fp; ner_fn += fn

        scheduler.step()

        sp = torch.cat(all_sp); sg = torch.cat(all_sg)
        sent_acc = (sp == sg).float().mean().item()

        p  = ner_tp / (ner_tp + ner_fp + 1e-8)
        r  = ner_tp / (ner_tp + ner_fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)

        n_train = len(train_loader)
        n_val   = len(val_loader)

        avg_train_loss = train_loss/n_train
        avg_val_loss   = val_loss/n_val

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"Loss {avg_train_loss:.3f}→{avg_val_loss:.3f} | "
              f"NER F1 {f1:.3f} | Sent Acc {sent_acc:.3f}", end="")

        score = f1 * 0.5 + sent_acc * 0.5
        if best_state is None or score > best_score:
            best_score = score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(" ✓", end="")
        print()

    model.load_state_dict(best_state)
    print(f"\nBest score: {best_score:.3f}")

    plt.figure(figsize=(8,5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training / Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return model

### Inference

In [10]:
def predict(text, model, vocab):
    tokens  = tokenize(text)[:MAX_SEQ_LEN]
    tok_ids = torch.tensor([vocab.encode(tokens)], dtype=torch.long).to(DEVICE)
    lengths = torch.tensor([len(tokens)])

    model.eval()
    with torch.no_grad():
        # Viterbi decode cho NER
        paths       = model.decode(tok_ids, lengths)     # [[tag_id, ...]]
        bio         = [NER_LABELS[i] for i in paths[0]]

        # Sentiment qua forward bình thường
        _, sent_logits = model(tok_ids, lengths)

    # Trích xuất entity spans từ chuỗi BIO
    entities, i = [], 0
    while i < len(bio):
        if bio[i].startswith("B-"):
            typ, j = bio[i][2:], i + 1
            while j < len(bio) and bio[j] == f"I-{typ}":
                j += 1
            entities.append({"text": " ".join(tokens[i:j]), "type": typ})
            i = j
        else:
            i += 1

    sent_idx   = sent_logits[0].argmax().item()
    sent_label = SENT_LABELS[sent_idx]
    sent_probs = F.softmax(sent_logits[0], dim=-1).tolist()
    score      = sum((i - 1) * p for i, p in enumerate(sent_probs))

    return {
        "text"     : text,
        "entities" : entities,
        "sentiment": sent_label,
        "score"    : round(score, 3),
    }


def show(result):
    SMAP = {"pos": "✅", "neg": "❌", "neu": "⚪"}
    entities_display = [
        {"text": e["text"].replace("_", " "), "type": e["type"]}
        for e in result["entities"]
    ]
    print(f"\n📝 {result['text']}")
    print(f"   Entities : {entities_display or '—'}")
    print(f"   Sentiment: {SMAP.get(result['sentiment'], '')} "
          f"{result['sentiment']} (score={result['score']:+.2f})")


### Evaluate

In [11]:
def evaluate(test_path, model, vocab):
    print(f"\n── Evaluate: {test_path} ──")
    test_data = load_data(test_path)

    test_loader = DataLoader(
        NERSentDataset(test_data, vocab),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )

    sent_loss_fn = nn.CrossEntropyLoss()
    model.eval()

    all_sp, all_sg = [], []
    ner_tp = ner_fp = ner_fn = 0
    total_loss = 0.0

    with torch.no_grad():
        for tok, ner, sent, lens in test_loader:
            tok, ner, sent = tok.to(DEVICE), ner.to(DEVICE), sent.to(DEVICE)

            emissions, sent_logits = model(tok, lens)

            safe_ner = ner.clone()
            safe_ner[safe_ner == -100] = 0
            ner_loss   = model.crf.neg_log_likelihood(emissions, safe_ner, lens)
            sent_loss  = sent_loss_fn(sent_logits, sent)
            total_loss += (ner_loss + sent_loss).item()

            all_sp.append(sent_logits.argmax(-1).cpu())
            all_sg.append(sent.cpu())

            pred_paths  = model.crf.decode(emissions, lens)
            pred_tensor = torch.zeros_like(ner)
            for b, path in enumerate(pred_paths):
                pred_tensor[b, :len(path)] = torch.tensor(path)
            tp, fp, fn = ner_counts(pred_tensor.cpu(), ner.cpu())
            ner_tp += tp; ner_fp += fp; ner_fn += fn

    sp = torch.cat(all_sp); sg = torch.cat(all_sg)
    sent_acc = (sp == sg).float().mean().item()

    p  = ner_tp / (ner_tp + ner_fp + 1e-8)
    r  = ner_tp / (ner_tp + ner_fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)

    print(f"Loss     : {total_loss / len(test_loader):.4f}")
    print(f"NER F1   : {f1:.4f}  (P={p:.4f} R={r:.4f})")
    print(f"Sent Acc : {sent_acc:.4f}")

    # Chi tiết sentiment
    from collections import Counter
    sent_l2i = {l: i for i, l in enumerate(SENT_LABELS)}
    correct  = Counter()
    total    = Counter()
    for pred, gold in zip(sp.tolist(), sg.tolist()):
        label = SENT_LABELS[gold]
        total[label]   += 1
        if pred == gold:
            correct[label] += 1
    print("Sentiment per class:")
    for label in SENT_LABELS:
        n = total[label]
        c = correct[label]
        print(f"  {label:5s}: {c}/{n} ({c/n*100:.1f}%)" if n else f"  {label}: 0 samples")

    return {"ner_f1": f1, "sent_acc": sent_acc}


### Main

In [12]:
if __name__ == "__main__":

    # Load data
    all_data = load_data(DATA_PATH)
    random.shuffle(all_data)
    split      = int(0.8 * len(all_data))
    train_data = all_data[:split]
    val_data   = all_data[split:]
    print(f"Train: {len(train_data)}  Val: {len(val_data)}")

    # Vocab
    vocab = Vocabulary()
    vocab.build(train_data)

    # DataLoader
    train_loader = DataLoader(
        NERSentDataset(train_data, vocab),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        NERSentDataset(val_data, vocab),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )

    # Model
    model     = BiLSTMCRFNERSent(vocab_size=len(vocab)).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}\n")

    # Train
    model = train(EPOCHS, model, train_loader, val_loader, optimizer, scheduler)

    # Evaluate trên test set
    import os
    if os.path.exists(TEST_PATH):
        evaluate(TEST_PATH, model, vocab)
    else:
        print(f"\n[!] Không tìm thấy {TEST_PATH}, bỏ qua evaluate")


Loaded 13486 samples | Sentiment: {'pos': 3524, 'neg': 4646, 'neu': 5316}
Train: 10788  Val: 2698
Vocab size: 13727
Parameters: 2,451,900

Sent class weights: {'neg': '0.971', 'neu': '0.913', 'pos': '1.116'}
Epoch 01/30 | Loss 7.024→3.837 | NER F1 0.610 | Sent Acc 0.404 ✓
Epoch 02/30 | Loss 3.550→2.960 | NER F1 0.719 | Sent Acc 0.440 ✓
Epoch 03/30 | Loss 2.832→2.569 | NER F1 0.762 | Sent Acc 0.449 ✓
Epoch 04/30 | Loss 2.391→2.408 | NER F1 0.779 | Sent Acc 0.440 ✓
Epoch 05/30 | Loss 2.122→2.276 | NER F1 0.807 | Sent Acc 0.471 ✓
Epoch 06/30 | Loss 1.924→2.213 | NER F1 0.814 | Sent Acc 0.489 ✓
Epoch 07/30 | Loss 1.751→2.167 | NER F1 0.828 | Sent Acc 0.521 ✓
Epoch 08/30 | Loss 1.636→2.113 | NER F1 0.825 | Sent Acc 0.542 ✓
Epoch 09/30 | Loss 1.540→2.053 | NER F1 0.834 | Sent Acc 0.556 ✓


KeyboardInterrupt: 

In [ ]:
import pandas as pd
df = pd.read_json('test.json', lines=True)

texts = df['tokens'].apply(lambda x: " ".join(x).replace("_", " ")).tolist()

for t in range(len(texts)):
    if t <= 100:
        show(predict(texts[t], model, vocab))

c:\Users\phant\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📝 Nào ngồi xuống .
   Entities : —
   Sentiment: ⚪ neu (score=-0.17)

📝 Về văn hoá - xã hội : đẩy mạnh xã hội hoá , phát triển xã hội học tập , ưu tiên đầu tư ngân sách cho giáo dục - đào tạo .
   Entities : —
   Sentiment: ✅ pos (score=+0.60)

📝 Quy cách chuẩn được phổ biến của địa đạo là chiều rộng khoảng 9 tấc , chiều cao khoảng 1,1 m .
   Entities : —
   Sentiment: ⚪ neu (score=+0.15)

📝 Suốt đêm không ngủ , không muốn ngủ tí nào .
   Entities : —
   Sentiment: ❌ neg (score=-0.82)

📝 Là những người kháng chiến cũ , bố mẹ , dì tôi sẽ không lấy đó làm điều đau khổ đâu .
   Entities : —
   Sentiment: ✅ pos (score=+0.71)

📝 Cậu bé chỉ cho tôi chỗ thu mua bò cạp duy nhất tại đây : “ Ngay ngã ba , gần trụ sở uỷ ban xã , có treo tấm bảng lớn lắm ! ” .
   Entities : —
   Sentiment: ⚪ neu (score=+0.12)

📝 Thái độ không thể là đóng kịch được .
   Entities : —
   Sentiment: ❌ neg (score=-0.79)

📝 Lại một lần nữa , chính ban quản lí rừng phòng hộ đầu nguồn Sêrêpôk đã " thả cửa " cho sự hình t